# Project BayouIndex Workbook

In [2]:
# STEP 1: Load API key
from dotenv import load_dotenv
import os

# Specify your env file path explicitly
load_dotenv(dotenv_path="API_Keys.env")

API_KEY = os.getenv("BLS_API_KEY")

if not API_KEY:
    raise EnvironmentError("Missing BLS_API_KEY in API_Keys.env")

print("✅ API key loaded successfully.")


✅ API key loaded successfully.


In [7]:
# STEP 2: Test connection to BLS API
import requests
import json

BLS_ENDPOINT = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
TEST_SERIES_ID = "APUS37B74714"  # Gasoline prices, Houston metro
START_YEAR = "2020"
END_YEAR = "2024"

payload = {
    "seriesid": [TEST_SERIES_ID],
    "startyear": START_YEAR,
    "endyear": END_YEAR,
    "registrationkey": API_KEY,
    "catalog": True   # 👈 this tells the API to include metadata
}

headers = {"Content-Type": "application/json"}

response = requests.post(BLS_ENDPOINT, headers=headers, data=json.dumps(payload))
response.raise_for_status()

data = response.json()

if data.get("status") == "REQUEST_SUCCEEDED":
    print("✅ API request succeeded.")
else:
    print("⚠️ API did not report success:", data.get("message"))


✅ API request succeeded.


In [8]:
# STEP 3: Preview JSON structure
import pprint

pp = pprint.PrettyPrinter(indent=4, width=100)
pp.pprint(data)


{   'Results': {   'series': [   {   'catalog': {   'area': 'Houston-The Woodlands-Sugar Land, TX',
                                                    'item': 'Gasoline, unleaded regular, per '
                                                            'gallon/3.785 liters',
                                                    'measure_data_type': 'Gasoline, unleaded '
                                                                         'regular, per '
                                                                         'gallon/3.785 liters',
                                                    'series_id': 'APUS37B74714',
                                                    'series_title': 'Gasoline, unleaded regular, '
                                                                    'per gallon/3.785 liters in '
                                                                    'Houston-The Woodlands-Sugar '
                                                                 

In [12]:
# STEP 4: Flatten BLS JSON data into a preview DataFrame
import pandas as pd
import re

# Safely extract the first series in the response
series_data = data.get("Results", {}).get("series", [])[0]

# Extract metadata (if catalog info is included)
catalog = series_data.get("catalog", {})
title = catalog.get("series_title", "")
area = catalog.get("area", "")
item = catalog.get("item", "")

records = []

for obs in series_data.get("data", []):
    period = obs.get("period", "")
    # Skip non-monthly periods like 'M13' (annual) or invalid entries
    if not re.match(r"^M(0[1-9]|1[0-2])$", period):
        continue

    try:
        records.append({
            "SeriesID": series_data.get("seriesID"),
            "Year": int(obs["year"]),
            "Period": period,
            "Month": int(period[1:]),  # e.g., "M03" → 3
            "Value": float(obs["value"]),
            "Title": title,
            "Area": area,
            "Item": item
        })
    except (KeyError, ValueError, TypeError):
        # Skip bad or incomplete records gracefully
        continue

# Build DataFrame
df_preview = pd.DataFrame(records)

# Add Date column and sort cleanly
if not df_preview.empty:
    df_preview["Date"] = pd.to_datetime(
        dict(year=df_preview["Year"], month=df_preview["Month"], day=1)
    )
    df_preview = df_preview.sort_values(["SeriesID", "Date"]).reset_index(drop=True)

# Display first few rows
df_preview.head(10)


,SeriesID,Year,Period,Month,Value,Title,Area,Item,Date
0,APUS37B74714,2020,M01,1,2.189,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-01-01
1,APUS37B74714,2020,M02,2,2.032,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-02-01
2,APUS37B74714,2020,M03,3,1.997,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-03-01
3,APUS37B74714,2020,M04,4,1.593,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-04-01
4,APUS37B74714,2020,M05,5,1.503,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-05-01
5,APUS37B74714,2020,M06,6,1.679,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-06-01
6,APUS37B74714,2020,M07,7,1.787,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-07-01
7,APUS37B74714,2020,M08,8,1.727,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-08-01
8,APUS37B74714,2020,M09,9,1.747,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-09-01
9,APUS37B74714,2020,M10,10,1.707,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2020-10-01


In [ ]:
# Final code to retrive sample data - Single series
import json, requests, pandas as pd, re

BLS_ENDPOINT = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
SERIES_ID = "APUS37B74714"   # change/add more IDs as needed
START_YEAR, END_YEAR = "2020", "2024"

# --- 1) Request with catalog=True so metadata is included ---
payload = {
    "seriesid": [SERIES_ID],
    "startyear": START_YEAR,
    "endyear": END_YEAR,
    "registrationkey": API_KEY,
    "catalog": True
}
resp = requests.post(BLS_ENDPOINT, headers={"Content-Type":"application/json"}, data=json.dumps(payload), timeout=30)
resp.raise_for_status()
j = resp.json()
assert j.get("status") == "REQUEST_SUCCEEDED", j.get("message")

# --- 2) Normalize into a tidy DataFrame (M01..M12 only), with metadata ---
series = j["Results"]["series"][0]
catalog = series.get("catalog", {}) or {}

# metadata (with safe fallbacks)
title = catalog.get("series_title", "")
area  = catalog.get("area_name", catalog.get("area", ""))
item  = catalog.get("item_name", catalog.get("item", ""))

# fallback: parse area/item from title if missing
if (not area or not item) and title:
    m = re.search(r"^(?P<item>.+?)\s+in\s+(?P<area>.+?),\s+average price", title, flags=re.IGNORECASE)
    if m:
        item = item or m.group("item").strip()
        area = area or m.group("area").strip()

rows = []
for obs in series.get("data", []):
    period = obs.get("period", "")
    if not (period.startswith("M") and period != "M13"):  # monthly only
        continue
    rows.append({
        "SeriesID": series.get("seriesID"),
        "Year": int(obs["year"]),
        "Month": int(period[1:]),
        "Period": period,
        "Value": float(obs["value"]),
        "Title": title,
        "Area": area,
        "Item": item
    })

df = pd.DataFrame(rows)
if not df.empty:
    df["Date"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=1))
    df = df[["Date","Year","Month","Period","SeriesID","Title","Area","Item","Value"]]\
            .sort_values(["SeriesID","Date"]).reset_index(drop=True)

df.head(12)


,Date,Year,Month,Period,SeriesID,Title,Area,Item,Value
0,2020-01-01,2020,1,M01,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2.189
1,2020-02-01,2020,2,M02,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",2.032
2,2020-03-01,2020,3,M03,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",1.997
3,2020-04-01,2020,4,M04,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",1.593
4,2020-05-01,2020,5,M05,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",1.503
5,2020-06-01,2020,6,M06,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",1.679
6,2020-07-01,2020,7,M07,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",1.787
7,2020-08-01,2020,8,M08,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",1.727
8,2020-09-01,2020,9,M09,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",1.747
9,2020-10-01,2020,10,M10,APUS37B74714,"Gasoline, unleaded regular, per gallon/3.785 l...","Houston-The Woodlands-Sugar Land, TX","Gasoline, unleaded regular, per gallon/3.785 l...",1.707


**Takeaways**
1. **Load API Key** – securely load the BLS API key from the environment file.  
2. **Send API Request** – query the BLS Public API (v2) for a selected series ID and date range.  
3. **Inspect JSON Response** – preview raw data and metadata returned from the API.  
4. **Flatten JSON into DataFrame** – extract monthly records (M01–M12), convert numeric fields, and build a clean DataFrame with columns:
   - `SeriesID`, `Year`, `Month`, `Period`, `Value`, `Title`, `Area`, `Item`, `Date`.

**Note:**  
1. This notebook currently pulls data for a *single* BLS series as a sample (e.g., Houston gasoline prices).

**Next Steps:**
- Modify code to handle **multiple series IDs** in a single request or loop.  
- Refactor logic into **reusable functions** (e.g., fetch, clean, merge).  
- Create a **`.py` script** dedicated to data acquisition for modular use.

Im testing to see if my other workstation successfully connects to Git to do contributions

testing to see if im successfelly able to pull and push update from my laptop
